# Configurable Regressor Experiments

This notebook is organized so users can run experiments on their own data by editing only `CONFIG`.

Workflow:
1. Update paths and feature options in `CONFIG`.
2. Run all cells.
3. Check saved metrics/plots under `plots/<experiment_name>@pca<k>/`.


In [ ]:
%matplotlib inline

import json
import pandas as pd


## Main Config

Configure json file to run experiments on a different dataset.


In [ ]:
config_path = 'configs/default.json'
with open(config_path, "r") as f:
    CONFIG = json.load(f)
CONFIG

## Helpers

The reusable experiment code lives in `qualisr_lab.regressors`, so notebook and CLI runs stay in sync.


In [ ]:
from qualisr_lab.regressors import deep_update, run_experiment


## Run One Experiment

In [ ]:
run = run_experiment(CONFIG)
run["results"]


## Optional Batch Runs

Set overrides and run several experiments in one go.


In [ ]:
# Override config loaded before
BATCH_OVERRIDES = [
    # {
    #     "experiment_name": "nr_only",
    #     "features": {
    #         "include": ["nr"],
    #         "include_stats": False,
    #     },
    # },
    # {
    #     "experiment_name": "fr_vgg_resnet",
    #     "features": {
    #         "include": ["fr", "vgg", "resnet"],
    #         "fr_refs": ["rlfn", "span"],
    #     },
    # },
]

# Launch alternate configs
CONFIG_FILES = [
    # "configs/default.json"
]

# Launch every JSON config in these folders. Subfolders are included.
CONFIG_FOLDERS = [
    # "configs/
]

from pathlib import Path


def collect_config_paths(config_files, config_folders):
    paths = [Path(path) for path in config_files]
    for folder in config_folders:
        folder_path = Path(folder)
        paths.extend(sorted(folder_path.rglob("*.json")))

    unique_paths = []
    seen = set()
    for path in paths:
        key = path.as_posix()
        if key in seen:
            continue
        seen.add(key)
        unique_paths.append(path)
    return unique_paths

batch_results = []
for override in BATCH_OVERRIDES:
    cfg = deep_update(CONFIG, override)
    result = run_experiment(cfg)
    row = result["results"].copy()
    row.insert(0, "config_path", "<override>")
    row.insert(0, "experiment", cfg["experiment_name"])
    batch_results.append(row)

config_paths = collect_config_paths(CONFIG_FILES, CONFIG_FOLDERS)
print(f"Running {len(config_paths)} config file(s).")
for config_path in config_paths:
    with open(config_path, "r") as f:
        cfg = json.load(f)
    result = run_experiment(cfg)
    row = result["results"].copy()
    row.insert(0, "config_path", config_path.as_posix())
    row.insert(0, "experiment", cfg["experiment_name"])
    batch_results.append(row)

if batch_results:
    summary = pd.concat(batch_results, ignore_index=True)
    display(summary)
else:
    print("No batch overrides configured.")
